# AE-TFPE — Stage B Evaluation (Colab)Thin orchestration around the **frozen evaluation scripts already committed** to therepository. No scientific logic lives in this notebook: corruption parameters, seedrules, checkpoint-selection rules, metrics and statistical tests all come from therepo at a pinned commit.**Runs in a separate session from training.** `scientific/checkpoints/` is treated as**read only**; everything this notebook produces goes under a separate`evaluation/` namespace on Drive.Two frozen benchmarks are run for every verified checkpoint:| | Benchmark | Measures ||---|---|---|| **A** | Clean / Easy / Moderate / Hard | synthetic **augmentation** robustness || **B** | Controlled Synthetic Corruption Benchmark (6 families x 3 severities) | targeted **corruption / noise** robustness |They are reported separately and never merged.**This notebook does not measure latency, throughput or memory.** Those come from thefrozen Tesla-T4 hardware benchmark and must not be replaced by numbers from anevaluation session on whatever GPU Colab allocates.**Run All is safe and resumable.** Completed (model, distribution) pairs are skipped.

## 0 — Configuration and the SAFE PARALLEL EXECUTION check

In [ ]:
# ---------------------------------------------------------------- configuration ----
REPO_URL    = "https://github.com/ducthong-dev/VisionTransformer-X-YOLO.git"
REPO_COMMIT = "b72e73f"        # frozen commit to evaluate at; set to a full SHA if you prefer
DRIVE_BASE  = "/content/drive/MyDrive/AE_TFPE_MajorRevision"

# READ ONLY. The training session owns this tree.
TRAIN_ROOT  = f"{DRIVE_BASE}/scientific"
CKPT_ROOT   = f"{TRAIN_ROOT}/checkpoints"

# This notebook's namespace. Nothing else writes here.
EVAL_ROOT   = f"{DRIVE_BASE}/evaluation"
EVAL_DIRS   = ["checkpoint_verification", "benchmark_a", "controlled_corruptions",
               "predictions", "statistics", "tables", "logs", "manifests"]

# Local Colab SSD. All inference reads and writes here; Drive is only mirrored to.
WORK        = "/content/eval_work"
LOCAL_CKPT  = f"{WORK}/checkpoints"
LOCAL_OUT   = f"{WORK}/out"
LOCAL_DATA  = "/content/data/Plant_leaf_diseases_dataset"
REPO_DIR    = "/content/VisionTransformer-X-YOLO"

# Dataset archive on Drive (3.25 GB, contains train/val/test + the three augmented sets).
DATASET_ZIP_CANDIDATES = [
    f"{DRIVE_BASE}/Plant_leaf_diseases_dataset_with_augment.zip",
    "/content/drive/MyDrive/Plant_leaf_diseases_dataset_with_augment.zip",
    "/content/drive/MyDrive/dataset/Plant_leaf_diseases_dataset_with_augment.zip",
]

BATCH_SIZE, NUM_WORKERS = 64, 2
FORCE_RECOMPUTE = False        # set True only to deliberately redo completed pairs

# ------------------------------------------------- SAFE PARALLEL EXECUTION check ----
import os

def _norm(p): return os.path.normpath(p).rstrip("/") + "/"

def safe_parallel_check():
    t, e, c = _norm(TRAIN_ROOT), _norm(EVAL_ROOT), _norm(CKPT_ROOT)
    problems = []
    if t == e:                      problems.append("training root == evaluation root")
    if e.startswith(t):             problems.append(f"evaluation root {e} is INSIDE training root {t}")
    if t.startswith(e):             problems.append(f"training root {t} is INSIDE evaluation root {e}")
    if e.startswith(c) or c.startswith(e):
        problems.append(f"evaluation root and checkpoint root {c} overlap")
    if _norm(LOCAL_OUT).startswith(_norm(LOCAL_CKPT)):
        problems.append("local output dir is inside the local checkpoint dir")
    if problems:
        raise SystemExit("REFUSING TO RUN - output roots overlap:\n  " + "\n  ".join(problems))
    print("SAFE PARALLEL EXECUTION: OK")
    print(f"  training  (READ ONLY) : {t}")
    print(f"  checkpoints (READ ONLY): {c}")
    print(f"  evaluation (write)    : {e}")
    print("  the two roots are disjoint; this session cannot disturb training output")

safe_parallel_check()

# Guard used by every copy in this notebook: nothing may ever be written under the
# training tree, whatever a later edit to this notebook does.
def assert_not_training_path(dst):
    d = _norm(os.path.abspath(dst))
    if d.startswith(_norm(TRAIN_ROOT)):
        raise PermissionError(f"BLOCKED: attempt to write inside the read-only training tree: {dst}")
    return dst

## 1 — GPU, Drive mount, evaluation namespace

In [ ]:
import subprocess, sys, json, time, shutil, hashlib
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

assert os.path.isdir(CKPT_ROOT), f"checkpoint root not found: {CKPT_ROOT}"
for d in EVAL_DIRS:
    os.makedirs(assert_not_training_path(f"{EVAL_ROOT}/{d}"), exist_ok=True)
for d in (WORK, LOCAL_CKPT, LOCAL_OUT, os.path.dirname(LOCAL_DATA)):
    os.makedirs(d, exist_ok=True)

print("\nevaluation namespace:")
for d in EVAL_DIRS:
    print(f"  {EVAL_ROOT}/{d}")
print(f"\ncheckpoint runs visible on Drive: {sorted(os.listdir(CKPT_ROOT))}")

## 2 — Repository at the frozen commit, and full provenance

In [ ]:
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git","clone","--quiet",REPO_URL,REPO_DIR], check=True)
subprocess.run(["git","-C",REPO_DIR,"fetch","--quiet","--all"], check=False)
subprocess.run(["git","-C",REPO_DIR,"checkout","--quiet",REPO_COMMIT], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

def sh(*a): return subprocess.run(a, capture_output=True, text=True).stdout.strip()

import torch, numpy, PIL, pandas, scipy
PROVENANCE = {
    "notebook": "AE_TFPE_StageB_Evaluation_Colab.ipynb",
    "purpose": "accuracy / robustness / predictions / statistics -- NOT hardware efficiency",
    "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "repo_url": REPO_URL,
    "git_commit": sh("git","-C",REPO_DIR,"rev-parse","HEAD"),
    "git_commit_short": sh("git","-C",REPO_DIR,"rev-parse","--short","HEAD"),
    "git_dirty": bool(sh("git","-C",REPO_DIR,"status","--porcelain")),
    "git_status": sh("git","-C",REPO_DIR,"status","--porcelain"),
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_memory_gb": (round(torch.cuda.get_device_properties(0).total_memory/1e9,2)
                      if torch.cuda.is_available() else None),
    "packages": {"numpy":numpy.__version__, "pillow":PIL.__version__,
                 "pandas":pandas.__version__, "scipy":scipy.__version__,
                 "torchvision":__import__("torchvision").__version__},
}
try:
    import ultralytics, timm, transformers
    PROVENANCE["packages"].update({"ultralytics":ultralytics.__version__,
                                   "timm":timm.__version__, "transformers":transformers.__version__})
except Exception as e:
    print("NOTE: model libraries missing, installing below:", e)

assert torch.cuda.is_available(), "CUDA required. Runtime > Change runtime type > GPU (T4 or L4 is fine)."
if PROVENANCE["git_dirty"]:
    print("WARNING: repository working tree is DIRTY; provenance records the exact diff state.")
print(json.dumps(PROVENANCE, indent=2))

In [ ]:
# Dependencies. Pinned to what the frozen scripts expect; skipped if already satisfied.
need = []
try: import ultralytics
except ImportError: need.append("ultralytics==8.4.120")
try: import timm
except ImportError: need.append("timm")
try: import transformers
except ImportError: need.append("transformers==4.37.2")
try: import thop
except ImportError: need.append("thop")
if need:
    subprocess.run([sys.executable,"-m","pip","install","-q",*need], check=True)
    print("installed:", need, "\n>>> if this is the first run, RESTART THE RUNTIME and Run All again")
else:
    print("all model libraries present")

import importlib
for m in ("ultralytics","timm","transformers"):
    PROVENANCE["packages"][m] = importlib.import_module(m).__version__

with open(assert_not_training_path(f"{EVAL_ROOT}/manifests/session_provenance.json"), "w") as fh:
    json.dump(PROVENANCE, fh, indent=2)
print("provenance written to", f"{EVAL_ROOT}/manifests/session_provenance.json")

## 3 — Dataset: stage to local SSD and verify against the frozen hashes (hard fail)

In [ ]:
# Frozen integrity constants. These are the listings the A100 training runs recorded and
# the ones every result in this campaign refers to.
FROZEN = {
    "train": "6dc760b424a5deb9e9935ae6e5a13a4948a7d19d5b6de32e09e05ce57ee864bb",
    "val":   "733a495f83c029c0c94b8a780aa3c8d57bbb1fa1491532da59236088d659ec04",
    "test":  "aad05ffc1a693bc78444a0472d3cc17b46e948c92241a25d1bb345969ceef5d3",
    # the three augmented sets share a listing hash by construction (identical filenames)
    "augmented_test_images_easy":     "1a447413345ee86af4e107681bfbacc4c0cad62998734f022dd7bf2e6c19c586",
    "augmented_test_images_enhanced": "1a447413345ee86af4e107681bfbacc4c0cad62998734f022dd7bf2e6c19c586",
    "augmented_test_images_hardest":  "1a447413345ee86af4e107681bfbacc4c0cad62998734f022dd7bf2e6c19c586",
}
FROZEN_COUNTS = {"train": 38584, "val": 8346, "test": 8335,
                 "augmented_test_images_easy": 8335,
                 "augmented_test_images_enhanced": 8335,
                 "augmented_test_images_hardest": 8335}
DATASET_SHA256 = "4f8a8332c3900e318f172c633a2aa5ec8b475174b76152a9db828173bae1897d"

if not os.path.isdir(os.path.join(LOCAL_DATA, "test")):
    zip_path = next((p for p in DATASET_ZIP_CANDIDATES if os.path.exists(p)), None)
    if zip_path is None:
        raise SystemExit("Dataset archive not found. Set DATASET_ZIP_CANDIDATES to its Drive path.\n"
                         "Candidates tried:\n  " + "\n  ".join(DATASET_ZIP_CANDIDATES))
    print(f"unzipping {zip_path} -> {os.path.dirname(LOCAL_DATA)} (few minutes, once per session)")
    subprocess.run(["unzip","-q","-o",zip_path,"-d",os.path.dirname(LOCAL_DATA)], check=True)
else:
    print("dataset already staged on local SSD")

from aetfpe.data import dataset_fingerprint, list_classes   # frozen implementation
from aetfpe.provenance import dataset_identity

fps, problems = {}, []
for split, n in FROZEN_COUNTS.items():
    root = os.path.join(LOCAL_DATA, split)
    if not os.path.isdir(root):
        problems.append(f"{split}: MISSING at {root}"); continue
    fp = dataset_fingerprint(root, list_classes(root)); fps[split] = fp
    if fp["num_images"] != n:
        problems.append(f"{split}: {fp['num_images']} images, expected {n}")
    if fp["num_classes"] != 39:
        problems.append(f"{split}: {fp['num_classes']} classes, expected 39")
    if fp["listing_sha256"] != FROZEN[split]:
        problems.append(f"{split}: listing sha {fp['listing_sha256'][:16]}... != frozen {FROZEN[split][:16]}...")
    print(f"  {split:32s} n={fp['num_images']:6d} c={fp['num_classes']:3d} {fp['listing_sha256'][:16]}...")

ds_id = dataset_identity(fps.get("train", {}), fps.get("val", {}))
if ds_id["sha256"] != DATASET_SHA256:
    problems.append(f"dataset_sha256 {ds_id['sha256'][:16]}... != training record {DATASET_SHA256[:16]}...")

if problems:
    raise SystemExit("DATASET VERIFICATION FAILED - refusing to evaluate:\n  " + "\n  ".join(problems))
print(f"\nDATASET OK - dataset_sha256 {ds_id['sha256'][:24]}... matches the A100 training provenance")
json.dump({"fingerprints": fps, "dataset_identity": ds_id},
          open(assert_not_training_path(f"{EVAL_ROOT}/manifests/dataset_verification.json"),"w"), indent=2)

## 4 — Verify the four distributions and the FROZEN clean↔augmented mapping

In [ ]:
# The mapping follows os.listdir order, which differs between filesystems. The committed
# artifact docs/evidence/clean_augmented_mapping.json is AUTHORITATIVE and must never be
# rebuilt here -- a Colab/ext4 rebuild would silently produce a different, wrong mapping.
MAPPING = f"{REPO_DIR}/docs/evidence/clean_augmented_mapping.json"
assert os.path.exists(MAPPING), "frozen mapping missing from the repo checkout"

r = subprocess.run([sys.executable,"scripts/verify_eval_distributions.py",
                    "--data-root",LOCAL_DATA,"--out",f"{LOCAL_OUT}/eval_integrity",
                    "--frozen-mapping",MAPPING,"--verify-limit","200"],
                   capture_output=True, text=True)
print(r.stdout[-4000:]); print(r.stderr[-2000:])
if r.returncode != 0:
    raise SystemExit("distribution verification FAILED - refusing to evaluate")

_m = json.load(open(MAPPING))
print(f"\nfrozen mapping: {len(_m['map'])} entries, verdict={_m['verdict']}")
shutil.copy(f"{LOCAL_OUT}/eval_integrity/distribution_verification.json",
            assert_not_training_path(f"{EVAL_ROOT}/manifests/distribution_verification.json"))

## 5 — Controlled Synthetic Corruption Benchmark: rebuild here and compare to the reference digests

In [ ]:
# The benchmark is applied on the fly. Rebuilding the manifest on this machine takes a
# couple of minutes and lets us prove the corruption arithmetic reproduces. JPEG is the
# one family whose pixels may legitimately differ across libjpeg builds.
r = subprocess.run([sys.executable,"scripts/generate_controlled_corruptions.py",
                    "--data-root",LOCAL_DATA,"--out",f"{LOCAL_OUT}/controlled_corruptions","--hashes"],
                   capture_output=True, text=True)
print(r.stdout[-2500:]); print(r.stderr[-1500:])
if r.returncode != 0:
    raise SystemExit("controlled-corruption freeze FAILED")

import gzip, csv, collections
ref = json.load(open(f"{REPO_DIR}/docs/evidence/controlled_corruption_reference.json"))
per = collections.defaultdict(list)
with gzip.open(f"{LOCAL_OUT}/controlled_corruptions/controlled_corruption_manifest.csv.gz","rt") as fh:
    for row in csv.DictReader(fh):
        per[f"{row['family']}/{row['severity']}"].append((row["relative_path"], row["pixel_sha256"]))

match, differ = [], []
for k, v in sorted(per.items()):
    h = hashlib.sha256()
    for p, x in sorted(v): h.update(p.encode()); h.update(x.encode())
    (match if h.hexdigest() == ref["digests"][k]["digest"] else differ).append(k)

print(f"\nfamily digests reproducing the macOS reference : {len(match)}/{len(per)}")
for k in differ: print(f"  DIFFERS: {k}")
non_jpeg = [k for k in differ if not k.startswith("jpeg/")]
if non_jpeg:
    raise SystemExit(f"REPRODUCIBILITY FAILURE in non-JPEG families: {non_jpeg}")
if differ:
    print("  (JPEG-only divergence is expected across libjpeg builds and is recorded, not a defect.\n"
          "   All models in this session share the identical JPEG images, so the comparison stays fair.)")

CC_REPRO = {"matched": match, "differed": differ, "jpeg_only_divergence": bool(differ) and not non_jpeg,
            "reference_environment": ref["reference_environment"], "this_environment": PROVENANCE["packages"]}
json.dump(CC_REPRO, open(assert_not_training_path(f"{EVAL_ROOT}/manifests/controlled_corruption_reproducibility.json"),"w"), indent=2)
for f in ("controlled_corruption_spec.json",):
    shutil.copy(f"{LOCAL_OUT}/controlled_corruptions/{f}",
                assert_not_training_path(f"{EVAL_ROOT}/controlled_corruptions/{f}"))

## 6 — Read the campaign manifest, decide which runs are VALID, build the waves

In [ ]:
# Frozen wave order. B1/B3 are appended ONLY if the manifest later shows them COMPLETED.
WAVE_1 = ["A0", "A5", "D1", "E5", "B2"]
WAVE_2 = ["A1", "A2", "A3", "A4"]
WAVE_3 = ["F2", "F4", "F1", "E3", "E7", "M1", "M2", "M3"]
CONDITIONAL = ["B1", "B3"]
ALIASES = {"F3": "A3", "F5": "A5", "F5_clean": "D1"}     # never evaluated separately

man_path = f"{TRAIN_ROOT}/campaign_manifest.json"
manifest = json.load(open(man_path))                      # READ ONLY
runs = manifest["runs"]

def is_complete(rid):
    r = runs.get(rid, {})
    if r.get("status") != "COMPLETED":
        return False, r.get("status", "ABSENT")
    s_path = f"{CKPT_ROOT}/{rid}/train_summary.json"
    if not os.path.exists(s_path):
        return False, "no train_summary.json"
    s = json.load(open(s_path))                            # READ ONLY
    if s.get("status") != "completed":  return False, f"summary status {s.get('status')}"
    if s.get("epochs_completed") != 50: return False, f"{s.get('epochs_completed')}/50 epochs"
    return True, "COMPLETED 50/50"

print(f"{'run':10s} {'manifest':14s} {'gate':22s} wave")
waves = {"wave1": [], "wave2": [], "wave3": []}
for wname, wl in (("wave1",WAVE_1), ("wave2",WAVE_2), ("wave3",WAVE_3)):
    for rid in wl:
        ok, why = is_complete(rid)
        if ok: waves[wname].append(rid)
        print(f"{rid:10s} {runs.get(rid,{}).get('status','ABSENT'):14s} {why:22s} {wname if ok else 'EXCLUDED'}")
for rid in CONDITIONAL:
    ok, why = is_complete(rid)
    if ok:
        waves["wave3"].append(rid)
        print(f"{rid:10s} {runs.get(rid,{}).get('status','ABSENT'):14s} {why:22s} wave3 (appended: now VALID)")
    else:
        print(f"{rid:10s} {runs.get(rid,{}).get('status','ABSENT'):14s} {why:22s} NOT VALID - skipped")

ALL_RUNS = waves["wave1"] + waves["wave2"] + waves["wave3"]
print(f"\nevaluable: {len(ALL_RUNS)} runs -> {ALL_RUNS}")
json.dump({"waves": waves, "all_runs": ALL_RUNS, "manifest_read": man_path},
          open(assert_not_training_path(f"{EVAL_ROOT}/manifests/wave_plan.json"),"w"), indent=2)

## 7 — Stage checkpoints to local SSD (read-only source, never `last.pt`) and run the frozen gate

In [ ]:
STAGE_FILES = ["checkpoint.pt", "run_provenance.json", "train_summary.json", "metrics.csv"]
NEVER_COPY  = ["last.pt"]        # optimiser state; not needed for inference, and large

def stage(rid):
    src, dst = f"{CKPT_ROOT}/{rid}", f"{LOCAL_CKPT}/{rid}"
    os.makedirs(assert_not_training_path(dst), exist_ok=True)
    got = []
    for f in STAGE_FILES:
        s, d = f"{src}/{f}", f"{dst}/{f}"
        if os.path.exists(s) and not os.path.exists(d):
            t0 = time.time(); shutil.copy2(s, assert_not_training_path(d))
            got.append(f"{f} ({os.path.getsize(d)/1e6:.0f} MB, {time.time()-t0:.0f}s)")
        elif os.path.exists(d):
            got.append(f"{f} (cached)")
    return got

for rid in ALL_RUNS:
    print(f"[{rid}] " + ", ".join(stage(rid)))
assert not any(os.path.exists(f"{LOCAL_CKPT}/{r}/last.pt") for r in ALL_RUNS), "last.pt must never be staged"
print("\nstaged to local SSD; last.pt never copied")

In [ ]:
# The frozen gate. Refuses smoke / wrong-namespace / wrong-dataset / non-50-epoch
# artifacts, and confirms each checkpoint really is its run's best_val_top1 selection.
VERIF = f"{LOCAL_OUT}/checkpoint_verification.json"
r = subprocess.run([sys.executable,"scripts/verify_checkpoints.py","--root",LOCAL_CKPT,
                    "--out",VERIF,"--data-root",LOCAL_DATA], capture_output=True, text=True)
print(r.stdout[-6000:]); print(r.stderr[-2000:])

ver = json.load(open(VERIF))
ACCEPTED, REFUSED = ver["accepted"], ver["refused"]
print(f"\nACCEPTED {len(ACCEPTED)}: {ACCEPTED}")
if REFUSED: print(f"REFUSED  {len(REFUSED)}: {REFUSED}   <-- reported, never silently skipped")
assert ACCEPTED, "no checkpoint passed the gate; nothing may be evaluated"

for w in waves: waves[w] = [r_ for r_ in waves[w] if r_ in ACCEPTED]
ALL_RUNS = waves["wave1"] + waves["wave2"] + waves["wave3"]
shutil.copy(VERIF, assert_not_training_path(f"{EVAL_ROOT}/checkpoint_verification/checkpoint_verification.json"))
print(f"\nfinal evaluation set: {ALL_RUNS}")

## 8 — Resume: pull completed outputs back from Drive, and the progress dashboard

In [ ]:
BENCH_A_LOCAL, BENCH_B_LOCAL = f"{LOCAL_OUT}/evaluation", f"{LOCAL_OUT}/evaluation_controlled"
BENCH_A_DRIVE, BENCH_B_DRIVE = f"{EVAL_ROOT}/benchmark_a", f"{EVAL_ROOT}/controlled_corruptions"

DISTS_A = ["clean", "easy", "moderate", "hard"]
import yaml as _yaml
_cc = _yaml.safe_load(open("configs/controlled_corruptions.yaml"))
DISTS_B = ["clean_none"] + [f"{f}_{s}" for f, sv in _cc["corruptions"].items() for s in sv]

def restore_from_drive():
    """A completed artifact on Drive means that pair never needs recomputing."""
    n = 0
    for drive_root, local_root in ((BENCH_A_DRIVE, BENCH_A_LOCAL), (BENCH_B_DRIVE, BENCH_B_LOCAL)):
        if not os.path.isdir(drive_root): continue
        for rid in sorted(os.listdir(drive_root)):
            if not os.path.isdir(f"{drive_root}/{rid}"): continue
            os.makedirs(assert_not_training_path(f"{local_root}/{rid}"), exist_ok=True)
            for f in os.listdir(f"{drive_root}/{rid}"):
                d = f"{local_root}/{rid}/{f}"
                if not os.path.exists(d):
                    shutil.copy2(f"{drive_root}/{rid}/{f}", assert_not_training_path(d)); n += 1
    return n

print(f"restored {restore_from_drive()} completed artifacts from Drive")

START = time.time()
def progress():
    doneA = {r: sum(os.path.exists(f"{BENCH_A_LOCAL}/{r}/predictions_{d}.csv.gz") for d in DISTS_A)
             for r in ALL_RUNS}
    doneB = {r: sum(os.path.exists(f"{BENCH_B_LOCAL}/{r}/predictions_{d}.csv.gz") for d in DISTS_B)
             for r in ALL_RUNS}
    tA, tB = len(DISTS_A)*len(ALL_RUNS), len(DISTS_B)*len(ALL_RUNS)
    cA, cB = sum(doneA.values()), sum(doneB.values())
    el = time.time() - START
    print("=" * 74)
    print(f"{'run':8s} {'A: clean/easy/mod/hard':24s} {'B: 19 controlled dists':24s} status")
    for r in ALL_RUNS:
        a, b = doneA[r], doneB[r]
        st = "complete" if (a == len(DISTS_A) and b == len(DISTS_B)) else ("in progress" if a or b else "pending")
        print(f"{r:8s} {'#'*a:<4s} {a}/{len(DISTS_A):<18d} {'#'*b:<19s} {b}/{len(DISTS_B):<4d} {st}")
    print("-" * 74)
    print(f"benchmark A {cA}/{tA} pairs   benchmark B {cB}/{tB} pairs   "
          f"total {cA+cB}/{tA+tB} ({100*(cA+cB)/max(tA+tB,1):.1f}%)")
    print(f"models complete: {sum(1 for r in ALL_RUNS if doneA[r]==len(DISTS_A) and doneB[r]==len(DISTS_B))}/{len(ALL_RUNS)}"
          f"   pending: {[r for r in ALL_RUNS if doneA[r]<len(DISTS_A) or doneB[r]<len(DISTS_B)]}")
    if REFUSED: print(f"refused checkpoints (excluded): {REFUSED}")
    print(f"elapsed this session {el/60:.1f} min", end="")
    if cA + cB: print(f"   ~{el/(cA+cB)*((tA+tB)-(cA+cB))/60:.0f} min remaining [rough]")
    else: print()
    print("=" * 74)

progress()

## 9 — Wave runner: evaluate on local SSD, then atomically mirror completed artifacts to Drive

In [ ]:
def mirror(local_root, drive_root, rid):
    """Copy completed artifacts to Drive via a temp name + rename, so a disconnect
    can never leave a half-written file that a later run would trust."""
    src, dst = f"{local_root}/{rid}", f"{drive_root}/{rid}"
    if not os.path.isdir(src): return 0
    os.makedirs(assert_not_training_path(dst), exist_ok=True)
    n = 0
    for f in sorted(os.listdir(src)):
        s, d = f"{src}/{f}", f"{dst}/{f}"
        if os.path.exists(d) and os.path.getsize(d) == os.path.getsize(s): continue
        tmp = d + ".part"
        shutil.copy2(s, assert_not_training_path(tmp)); os.replace(tmp, d); n += 1
    return n

def run_wave(name, run_ids):
    if not run_ids:
        print(f"[{name}] nothing to do"); return
    print(f"\n{'='*74}\nWAVE {name}: {run_ids}\n{'='*74}")
    for rid in run_ids:
        for tag, script, out_local, out_drive, extra in (
            ("A", "scripts/evaluate_distributions.py",        BENCH_A_LOCAL, BENCH_A_DRIVE,
             ["--mapping", MAPPING]),
            ("B", "scripts/evaluate_controlled_corruptions.py", BENCH_B_LOCAL, BENCH_B_DRIVE,
             ["--frozen", f"{LOCAL_OUT}/controlled_corruptions"]),
        ):
            cmd = [sys.executable, script, "--campaign-root", LOCAL_CKPT,
                   "--verification", VERIF, "--data-root", LOCAL_DATA,
                   "--out", out_local, "--runs", rid, "--device", "cuda",
                   "--batch-size", str(BATCH_SIZE), "--num-workers", str(NUM_WORKERS)] + extra
            if FORCE_RECOMPUTE: cmd.append("--force")
            t0 = time.time()
            r = subprocess.run(cmd, capture_output=True, text=True)
            print(f"[{rid}][benchmark {tag}] {time.time()-t0:.0f}s")
            print("   " + "\n   ".join(r.stdout.strip().splitlines()[-8:]))
            if r.returncode != 0:
                print("   STDERR:", r.stderr[-1500:])
                raise SystemExit(f"{rid} benchmark {tag} failed")
            print(f"   mirrored {mirror(out_local, out_drive, rid)} artifacts to Drive")
    progress()

def run_stats(label):
    """Frozen aggregation + statistics. Re-run after each wave; always safe to repeat."""
    jobs = [
        ("consolidated table", [sys.executable,"scripts/consolidated_evidence_table.py",
            "--eval-root",BENCH_A_LOCAL,
            "--out-csv",f"{LOCAL_OUT}/tables/consolidated_evidence_table.csv",
            "--out-md", f"{LOCAL_OUT}/tables/CONSOLIDATED_EVIDENCE_TABLE.md"]),
        ("paired statistics (A)", [sys.executable,"scripts/paired_statistics.py",
            "--eval-root",BENCH_A_LOCAL,"--out",f"{LOCAL_OUT}/statistics/paired_statistics.json"]),
        ("controlled report (B)", [sys.executable,"scripts/controlled_corruption_report.py",
            "--eval-root",BENCH_B_LOCAL,"--frozen",f"{LOCAL_OUT}/controlled_corruptions",
            "--out-json",f"{LOCAL_OUT}/statistics/controlled_report.json",
            "--out-md",  f"{LOCAL_OUT}/tables/CONTROLLED_CORRUPTION_RESULTS.md"]),
    ]
    os.makedirs(f"{LOCAL_OUT}/tables", exist_ok=True); os.makedirs(f"{LOCAL_OUT}/statistics", exist_ok=True)
    for nm, cmd in jobs:
        r = subprocess.run(cmd, capture_output=True, text=True)
        print(f"  {nm}: {'ok' if r.returncode==0 else 'FAILED'}")
        if r.returncode != 0: print("   ", r.stderr[-800:])
    for sub, dest in (("tables","tables"), ("statistics","statistics")):
        for f in sorted(os.listdir(f"{LOCAL_OUT}/{sub}")):
            tmp = f"{EVAL_ROOT}/{dest}/{f}.part"
            shutil.copy2(f"{LOCAL_OUT}/{sub}/{f}", assert_not_training_path(tmp))
            os.replace(tmp, f"{EVAL_ROOT}/{dest}/{f}")
    print(f"  mirrored tables + statistics to Drive  [{label}]")

## 10 — Wave 1: the decision-critical evidence checkpoint`A0 · A5 · D1 · E5 · B2` — baseline, Original, Efficient, denoising objective, externalbaseline. Includes the headline **A5 vs D1** comparison on both benchmarks.

In [ ]:
run_wave("1 (decision-critical)", waves["wave1"])
run_stats("after wave 1")

## 11 — Wave 2: completes the component ablation A0 → A1 → A2 → A3 → A4 → A5

In [ ]:
run_wave("2 (component ablation)", waves["wave2"])
run_stats("after wave 2")

## 12 — Wave 3: fusion comparators, mechanism controls, and any newly-VALID baseline

In [ ]:
run_wave("3 (remaining)", waves["wave3"])
run_stats("after wave 3")

## 13 — Final state and session manifest

In [ ]:
progress()
final = {
    "provenance": PROVENANCE,
    "waves": waves, "accepted": ACCEPTED, "refused": REFUSED,
    "benchmark_a_distributions": DISTS_A,
    "benchmark_b_distributions": DISTS_B,
    "controlled_corruption_reproducibility": CC_REPRO,
    "drive_paths": {"read_only_training": TRAIN_ROOT, "read_only_checkpoints": CKPT_ROOT,
                    "evaluation_namespace": EVAL_ROOT},
    "not_measured_here": ("latency, throughput and peak memory -- those come from the frozen "
                          "Tesla-T4 hardware benchmark and must not be replaced by numbers "
                          "from this session's GPU"),
    "seed_variability": "UNAVAILABLE (single training seed)",
    "finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
p = assert_not_training_path(f"{EVAL_ROOT}/manifests/evaluation_session.json")
json.dump(final, open(p,"w"), indent=2, default=str)
print("wrote", p)

# Prove the read-only contract held.
import subprocess as _sp
newest = _sp.run(["bash","-lc",
    f"find '{CKPT_ROOT}' -newermt '-2 hours' -type f 2>/dev/null | head -5"],
    capture_output=True, text=True).stdout.strip()
print("\nfiles modified under the training checkpoint tree in the last 2h:")
print("  " + (newest.replace(chr(10), chr(10)+"  ") if newest
              else "(none - read-only contract held; any listing here would be the TRAINING session)"))